# Bench2Drive results exploration

Quick notebook to inspect leaderboard results and aggregate infractions/scores.

Set `results_path` below to the JSON you want to analyse (defaults to the medium eval run).

In [77]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

import sys
from pathlib import Path

repo_root = (Path(__file__).resolve().parent.parent
            if '__file__' in globals()
            else (Path.cwd() / "..").resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [78]:
# Point this to any leaderboard results JSON
results_path = Path("../outputs/eval_bench2drive220_full/eval_bench2drive220_full.json")
with results_path.open() as f:
    data = json.load(f)

data.keys(), data.get("_checkpoint", {}).keys()

(dict_keys(['_checkpoint', 'entry_status', 'eligible', 'sensors', 'values', 'labels']),
 dict_keys(['global_record', 'progress', 'records']))

In [89]:
data["_checkpoint"]['global_record']['infractions']

{'collisions_layout': 0.175,
 'collisions_pedestrian': 0.219,
 'collisions_vehicle': 2.763,
 'red_light': 0.175,
 'stop_infraction': 0.0,
 'outside_route_lanes': 0.047,
 'min_speed_infractions': 180.904,
 'yield_emergency_vehicle_infractions': 0.219,
 'scenario_timeouts': 0.0,
 'route_dev': 0.0,
 'vehicle_blocked': 0.044,
 'route_timeout': 0.0}

In [101]:
data["_checkpoint"]["global_record"]["scores_mean"]

{'score_composed': 84.462461,
 'score_route': 97.887818,
 'score_penalty': 0.862587}

In [ ]:
# Flatten route records into a DataFrame
route_records = data["_checkpoint"]["records"]
records_df = pd.DataFrame(route_records)

# Expand nested dicts
scores_df = records_df.pop("scores").apply(pd.Series)
meta_df = records_df.pop("meta").apply(pd.Series)

# Count infractions per type (length of each list)
infractions_df = records_df.pop("infractions").apply(lambda d: {k: len(v) for k, v in d.items()}).apply(pd.Series).add_prefix("inf_")

routes = pd.concat([records_df, scores_df, meta_df, infractions_df], axis=1)
routes.drop(columns=["inf_min_speed_infractions"], inplace=True)
routes

,index,route_id,scenario_name,weather_id,save_name,status,num_infractions,town_name,score_route,score_penalty,score_composed,route_length,duration_game,duration_system,inf_collisions_layout,inf_collisions_pedestrian,inf_collisions_vehicle,inf_red_light,inf_stop_infraction,inf_outside_route_lanes,inf_yield_emergency_vehicle_infractions,inf_scenario_timeouts,inf_route_dev,inf_vehicle_blocked,inf_route_timeout
0,0,RouteScenario_1711_rep0,ParkingCutIn_1,15,RouteScenario_1711_rep0_Town12_ParkingCutIn_1_...,Completed,20,Town12,100.0,1.000000,100.000000,133.138,19.20,33.372,0,0,0,0,0,0,0,0,0,0,0
1,1,RouteScenario_1773_rep0,ParkedObstacle_1,25,RouteScenario_1773_rep0_Town12_ParkedObstacle_...,Completed,18,Town12,100.0,1.000000,100.000000,132.062,22.60,42.248,0,0,0,0,0,0,0,0,0,0,0
2,2,RouteScenario_1790_rep0,HazardAtSideLane_1,8,RouteScenario_1790_rep0_Town12_HazardAtSideLan...,Completed,21,Town12,100.0,1.000000,100.000000,119.119,29.00,62.433,0,0,0,0,0,0,0,0,0,0,0
3,3,RouteScenario_1792_rep0,HazardAtSideLane_1,18,RouteScenario_1792_rep0_Town12_HazardAtSideLan...,Completed,21,Town12,100.0,0.600000,60.000000,133.100,12.05,22.063,0,0,1,0,0,0,0,0,0,0,0
4,4,RouteScenario_1825_rep0,ConstructionObstacleTwoWays_1,25,RouteScenario_1825_rep0_Town12_ConstructionObs...,Completed,20,Town12,100.0,0.536394,53.639399,132.064,106.95,160.999,0,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,215,RouteScenario_28219_rep0,NonSignalizedJunctionLeftTurnEnterFlow_1,1,RouteScenario_28219_rep0_Town12_NonSignalizedJ...,Completed,19,Town12,100.0,0.600000,60.000000,80.698,11.20,17.991,0,0,1,0,0,0,0,0,0,0,0
216,216,RouteScenario_28229_rep0,NonSignalizedJunctionLeftTurnEnterFlow_1,12,RouteScenario_28229_rep0_Town12_NonSignalizedJ...,Completed,20,Town12,100.0,0.600000,60.000000,87.454,36.30,57.560,0,0,1,0,0,0,0,0,0,0,0
217,217,RouteScenario_28241_rep0,SignalizedJunctionLeftTurnEnterFlow_1,0,RouteScenario_28241_rep0_Town12_SignalizedJunc...,Completed,20,Town12,100.0,1.000000,100.000000,76.928,13.50,22.103,0,0,0,0,0,0,0,0,0,0,0
218,218,RouteScenario_28243_rep0,SignalizedJunctionLeftTurnEnterFlow_1,2,RouteScenario_28243_rep0_Town12_SignalizedJunc...,Completed,17,Town12,100.0,1.000000,100.000000,82.600,15.95,26.160,0,0,0,0,0,0,0,0,0,0,0


In [80]:
# Overall status and score summary
print("Statuses:\n", routes["status"].value_counts())
print("\nScore stats (composed/route/penalty):")
display(routes[["score_composed", "score_route", "score_penalty"]].astype(float).describe())

Statuses:
 status
Completed                     211
Failed - TickRuntime            8
Failed - Agent got blocked      1
Name: count, dtype: int64

Score stats (composed/route/penalty):


,score_composed,score_route,score_penalty
count,220.000000,220.000000,220.000000
mean,84.462461,97.887818,0.862587
std,23.648076,12.051579,0.211738
min,0.000000,0.000000,0.216000
25%,60.000000,100.000000,0.600000
50%,100.000000,100.000000,1.000000
75%,100.000000,100.000000,1.000000
max,100.000000,100.000000,1.000000


In [81]:
# Infraction counts per type (total and per-km)
inf_cols = [c for c in routes.columns if c.startswith("inf_")]
counts = routes[inf_cols].sum().sort_values(ascending=False)
print("Total infractions per type:\n", counts)

# routes["route_km"] = routes["route_length"] / 1000.0
# inf_per_km = (routes[inf_cols].div(routes["route_km"], axis=0)).mean().sort_values(ascending=False)
# print("\nAverage infractions per km:\n", inf_per_km)

Total infractions per type:
 inf_collisions_vehicle                     63
inf_outside_route_lanes                     7
inf_collisions_pedestrian                   5
inf_yield_emergency_vehicle_infractions     5
inf_collisions_layout                       4
inf_red_light                               4
inf_vehicle_blocked                         1
inf_stop_infraction                         0
inf_scenario_timeouts                       0
inf_route_dev                               0
inf_route_timeout                           0
dtype: int64


In [82]:
# Grouped views: by town and scenario name
def group_summary(df, by):
    grp = df.groupby(by)[["score_composed", "score_route", "score_penalty"]]
    return grp.agg(["mean", "std", "count"]).sort_values(("score_composed", "mean"), ascending=False)

print("Scores by town:")
display(group_summary(routes, "town_name"))

print("Scores by scenario:")
display(group_summary(routes, "scenario_name"))

Scores by town:


score_composed                  score_route                   \
                    mean        std count        mean        std count   
town_name                                                                
Town10HD      100.000000   0.000000     4  100.000000   0.000000     4   
Town05         89.102466  18.743968     9   98.686667   3.940000     9   
Town13         87.653802  19.963561    47  100.000000   0.000000    47   
Town01         87.500000  25.000000     4  100.000000   0.000000     4   
Town02         87.500000  25.000000     4  100.000000   0.000000     4   
Town12         86.159742  21.840534   104   98.442404   9.463946   104   
Town11         86.000000  24.467666     7  100.000000   0.000000     7   
Town07         82.136271  26.566602     5   91.530000  18.939496     5   
Town06         80.000000  21.908902     6  100.000000   0.000000     6   
Town04         74.947500  30.465360    12   98.814167   4.107847    12   
Town15         70.203714  28.631215     7   95.101429  12.960402     7   
Town03         65.090909  39.611752    11   81.818182  40.451992    11   

          score_penalty                  
                   mean       std count  
town_name                                
Town10HD       1.000000  0.000000     4  
Town05         0.904158  0.190823     9  
Town13         0.876538  0.199636    47  
Town01         0.875000  0.250000     4  
Town02         0.875000  0.250000     4  
Town12         0.873007  0.200695   104  
Town11         0.860000  0.244677     7  
Town07         0.881132  0.162779     5  
Town06         0.800000  0.219089     6  
Town04         0.761333  0.311932    12  
Town15         0.731429  0.258678     7  
Town03         0.832727  0.245320    11

Scores by scenario:


score_composed                   \
                                                      mean        std count   
scenario_name                                                                 
CrossingBicycleFlow_1                           100.000000   0.000000     5   
HazardAtSideLaneTwoWays_1                       100.000000   0.000000     5   
ControlLoss_1                                   100.000000   0.000000     5   
VehicleTurningRoutePedestrian_1                 100.000000   0.000000     5   
T_Junction_1                                    100.000000   0.000000     5   
VehicleTurningRoute_1                           100.000000   0.000000     5   
VanillaNonSignalizedTurn_1                      100.000000   0.000000     5   
VanillaNonSignalizedTurnEncounterStopsign_1     100.000000   0.000000     5   
ParkingCutIn_1                                  100.000000   0.000000     5   
ParkedObstacleTwoWays_1                         100.000000   0.000000     5   
InterurbanAdvancedActorFlow_1                   100.000000   0.000000     5   
HighwayCutIn_1                                  100.000000   0.000000     5   
InvadingTurn_1                                   99.798691   0.450140     5   
VanillaSignalizedTurnEncounterRedLight_1         94.000000  13.416408     5   
BlockedIntersection_1                            92.000000  17.888544     5   
NonSignalizedJunctionRightTurn_1                 92.000000  17.888544     5   
MergerIntoSlowTraffic_1                          92.000000  17.888544     5   
InterurbanActorFlow_1                            92.000000  17.888544     5   
VanillaSignalizedTurnEncounterGreenLight_1       88.136271  26.528104     5   
HardBreakRoute_1                                 87.200000  28.621670     5   
HazardAtSideLane_1                               86.790000  16.353018     5   
StaticCutIn_1                                    85.426000  32.588455     5   
SignalizedJunctionLeftTurnEnterFlow_1            84.000000  21.908902     5   
OppositeVehicleRunningRedLight_1                 84.000000  21.908902     5   
OppositeVehicleTakingPriority_1                  84.000000  21.908902     5   
EnterActorFlow_1                                 84.000000  21.908902     5   
ParkedObstacle_1                                 84.000000  21.908902     5   
PedestrianCrossing_1                             82.400000  26.053791     5   
ParkingCrossingPedestrian_1                      80.000000  27.386128     5   
ConstructionObstacleTwoWays_1                    76.727880  21.744812     5   
Accident_1                                       74.400000  35.054244     5   
SignalizedJunctionRightTurn_1                    72.400000  26.245000     5   
NonSignalizedJunctionLeftTurn_1                  71.800000  27.133006     5   
ConstructionObstacle_1                           71.520000  39.328768     5   
VehicleOpensDoorTwoWays_1                        71.200000  28.057085     5   
DynamicObjectCrossing_1                          70.000000  27.386128     5   
YieldToEmergencyVehicle_1                        70.000000   0.000000     5   
HighwayExit_1                                    68.052192  31.421427     5   
MergerIntoSlowTrafficV2_1                        68.000000  17.888544     5   
NonSignalizedJunctionLeftTurnEnterFlow_1         68.000000  17.888544     5   
AccidentTwoWays_1                                62.633639  22.515380     5   
SignalizedJunctionLeftTurn_1                     61.463600  25.775000     5   
ParkingExit_1                                    60.000000  54.772256     5   
SequentialLaneChange_1                           58.400000  26.168684     5   

                                            score_route                   \
                                                   mean        std count   
scenario_name                                                              
CrossingBicycleFlow_1                           100.000   0.000000     5   
HazardAtSideLaneTwoWays_1 

In [83]:
# Worst routes by composed score and by collision counts
print("Lowest composed scores:")
display(routes.sort_values("score_composed").head(10)[[
    "route_id", "scenario_name", "town_name", "status", "score_composed", "score_route", "score_penalty",
    "inf_collisions_vehicle", "inf_collisions_pedestrian", "inf_collisions_layout"
]])

routes["collisions_total"] = routes[["inf_collisions_vehicle", "inf_collisions_pedestrian", "inf_collisions_layout"]].sum(axis=1)
print("\nMost collisions:")
display(routes.sort_values("collisions_total", ascending=False).head(10)[[
    "route_id", "scenario_name", "town_name", "status", "collisions_total",
    "inf_collisions_vehicle", "inf_collisions_pedestrian", "inf_collisions_layout"
]])

Lowest composed scores:


,route_id,scenario_name,town_name,status,score_composed,score_route,score_penalty,inf_collisions_vehicle,inf_collisions_pedestrian,inf_collisions_layout
191,RouteScenario_26435_rep0,ParkingExit_1,Town03,Failed - TickRuntime,0.000000,0.00,1.000000,0,0,0
183,RouteScenario_26393_rep0,ParkingExit_1,Town03,Failed - TickRuntime,0.000000,0.00,1.000000,0,0,0
161,RouteScenario_24795_rep0,ConstructionObstacle_1,Town04,Completed,21.600000,100.00,0.216000,3,0,0
129,RouteScenario_23659_rep0,HighwayExit_1,Town12,Failed - TickRuntime,24.712514,65.35,0.378156,1,0,1
40,RouteScenario_2715_rep0,StaticCutIn_1,Town12,Failed - TickRuntime,27.130000,27.13,1.000000,0,0,0
105,RouteScenario_4468_rep0,SignalizedJunctionLeftTurn_1,Town12,Failed - TickRuntime,27.318000,45.53,0.600000,1,0,0
65,RouteScenario_3307_rep0,Accident_1,Town13,Completed,36.000000,100.00,0.360000,2,0,0
118,RouteScenario_17569_rep0,SequentialLaneChange_1,Town12,Completed,36.000000,100.00,0.360000,2,0,0
162,RouteScenario_24816_rep0,Accident_1,Town03,Completed,36.000000,100.00,0.360000,2,0,0
117,RouteScenario_17563_rep0,SequentialLaneChange_1,Town12,Completed,36.000000,100.00,0.360000,2,0,0



Most collisions:


,route_id,scenario_name,town_name,status,collisions_total,inf_collisions_vehicle,inf_collisions_pedestrian,inf_collisions_layout
161,RouteScenario_24795_rep0,ConstructionObstacle_1,Town04,Completed,3,3,0,0
160,RouteScenario_24785_rep0,ConstructionObstacle_1,Town04,Completed,2,2,0,0
75,RouteScenario_3476_rep0,VehicleOpensDoorTwoWays_1,Town13,Completed,2,2,0,0
162,RouteScenario_24816_rep0,Accident_1,Town03,Completed,2,2,0,0
65,RouteScenario_3307_rep0,Accident_1,Town13,Completed,2,2,0,0
12,RouteScenario_2091_rep0,NonSignalizedJunctionLeftTurn_1,Town12,Completed,2,1,0,1
129,RouteScenario_23659_rep0,HighwayExit_1,Town12,Failed - TickRuntime,2,1,0,1
117,RouteScenario_17563_rep0,SequentialLaneChange_1,Town12,Completed,2,2,0,0
118,RouteScenario_17569_rep0,SequentialLaneChange_1,Town12,Completed,2,2,0,0
188,RouteScenario_26406_rep0,HardBreakRoute_1,Town04,Completed,2,2,0,0


,index,route_id,scenario_name,weather_id,save_name,status,num_infractions,town_name,score_route,score_penalty,score_composed,route_length,duration_game,duration_system,inf_collisions_layout,inf_collisions_pedestrian,inf_collisions_vehicle,inf_red_light,inf_stop_infraction,inf_outside_route_lanes,inf_yield_emergency_vehicle_infractions,inf_scenario_timeouts,inf_route_dev,inf_vehicle_blocked,inf_route_timeout,collisions_total
0,0,RouteScenario_1711_rep0,ParkingCutIn_1,15,RouteScenario_1711_rep0_Town12_ParkingCutIn_1_...,Completed,20,Town12,100.0,1.000000,100.000000,133.138,19.20,33.372,0,0,0,0,0,0,0,0,0,0,0,0
1,1,RouteScenario_1773_rep0,ParkedObstacle_1,25,RouteScenario_1773_rep0_Town12_ParkedObstacle_...,Completed,18,Town12,100.0,1.000000,100.000000,132.062,22.60,42.248,0,0,0,0,0,0,0,0,0,0,0,0
2,2,RouteScenario_1790_rep0,HazardAtSideLane_1,8,RouteScenario_1790_rep0_Town12_HazardAtSideLan...,Completed,21,Town12,100.0,1.000000,100.000000,119.119,29.00,62.433,0,0,0,0,0,0,0,0,0,0,0,0
3,3,RouteScenario_1792_rep0,HazardAtSideLane_1,18,RouteScenario_1792_rep0_Town12_HazardAtSideLan...,Completed,21,Town12,100.0,0.600000,60.000000,133.100,12.05,22.063,0,0,1,0,0,0,0,0,0,0,0,1
4,4,RouteScenario_1825_rep0,ConstructionObstacleTwoWays_1,25,RouteScenario_1825_rep0_Town12_ConstructionObs...,Completed,20,Town12,100.0,0.536394,53.639399,132.064,106.95,160.999,0,0,1,0,0,1,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,215,RouteScenario_28219_rep0,NonSignalizedJunctionLeftTurnEnterFlow_1,1,RouteScenario_28219_rep0_Town12_NonSignalizedJ...,Completed,19,Town12,100.0,0.600000,60.000000,80.698,11.20,17.991,0,0,1,0,0,0,0,0,0,0,0,1
216,216,RouteScenario_28229_rep0,NonSignalizedJunctionLeftTurnEnterFlow_1,12,RouteScenario_28229_rep0_Town12_NonSignalizedJ...,Completed,20,Town12,100.0,0.600000,60.000000,87.454,36.30,57.560,0,0,1,0,0,0,0,0,0,0,0,1
217,217,RouteScenario_28241_rep0,SignalizedJunctionLeftTurnEnterFlow_1,0,RouteScenario_28241_rep0_Town12_SignalizedJunc...,Completed,20,Town12,100.0,1.000000,100.000000,76.928,13.50,22.103,0,0,0,0,0,0,0,0,0,0,0,0
218,218,RouteScenario_28243_rep0,SignalizedJunctionLeftTurnEnterFlow_1,2,RouteScenario_28243_rep0_Town12_SignalizedJunc...,Completed,17,Town12,100.0,1.000000,100.000000,82.600,15.95,26.160,0,0,0,0,0,0,0,0,0,0,0,0
